## АНД Предсказание следующего элемента
### Вариант 4: Улчшение работы RNN: Вариант RNN на Pytorch. Генерация поэзии с помощью нейронных сетей

В данном ноутбуке я представляю свою реализацию RNN на Pytorch, генерирующую стихи на основе романа в стихах "Евгений Онегин" Александра Сергеевича Пушкина.

Выполнил Жигалов Никита Сергеевич, 325гр.

Необходимые библиотеки

In [ ]:
import string
import os
from random import sample
import random
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F

from IPython.display import clear_output

import matplotlib.pyplot as plt
# %matplotlib inline

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("{} device is available".format(device))

#### 1. Загрузка данных.

In [ ]:
# Установка зависимостей выполняется отдельно по requirements.txt.

with open('onegin.txt', 'r') as iofile:
    text = iofile.readlines()

text = "".join([x.replace('\t\t', '').lower() for x in text])

#### 2. Построение словаря и предобработка текста
Хотим построить языковую модель на уровне символов. Приведем весь текст к нижнему регистру и построим словарь из всех символов в доступном корпусе текстов. Также добавим токен `<sos>`.

In [ ]:
tokens = sorted(set(text.lower())) + ["<sos>"]
num_tokens = len(tokens)

token_to_idx = {x: idx for idx, x in enumerate(tokens)}
idx_to_token = {idx: x for idx, x in enumerate(tokens)}

text_encoded = [token_to_idx[x] for x in text]

Вспомогательная функция, которая переводит строку в тензор из чисел, необходимо для корректного обучения

In [ ]:
# Вспомогательная функция превращающая строку в числа
def char_tensor(string):
    tensor = torch.zeros(len(string)).long()
    for c in range(len(string)):
        tensor[c] = tokens.index(string[c])
    return tensor

print(char_tensor('гусь'))

Сама сеть, ее слои:

1. `self.decoder = nn.Embedding(input_size, hidden_size)` - Слой эмбеддинга - переводим число, кодирующее входную последовательность символов в вектор скрытого состояния
2. `self.gru = nn.GRU(hidden_size, hidden_size, n_layers)` Собственно рекуррентный слой
3. `self.decoder = nn.Linear(hidden_size, output_size)` - Линейный слой, переводим вектора скрытах состояний в 

In [ ]:
import torch
import torch.nn as nn
from torch.autograd import Variable

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, n_layers=1):
        super(RNN, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        
        self.encoder = nn.Embedding(input_size, hidden_size) # входное кодирование слов
        self.gru = nn.GRU(hidden_size, hidden_size, n_layers) # собственно рекуррентный слой
        self.decoder = nn.Linear(hidden_size, output_size) # линейный выход
    
    def forward(self, input, hidden):
        input = self.encoder(input.view(1, -1)) # закодировали вход
        output, hidden = self.gru(input.view(1, 1, -1), hidden) # работа рекуррнтеого слоя
        output = self.decoder(output.view(1, -1)) # выход
        return output, hidden

    def init_hidden(self):
        return Variable(torch.zeros(self.n_layers, 1, self.hidden_size))

Вспомагательная функция, с помощью которой получаем один батч (случайный фрагмент текста по которому происходит обучение и оценка работы)

In [ ]:
chunk_len = 100
text_len = len(text)


def random_chunk():
    start_index = random.randint(0, text_len - chunk_len)
    end_index = start_index + chunk_len + 1
    return text[start_index:end_index]


print(random_chunk())

Так задаем обучающий фрагмент плюс фрагмент на котором будем оценивать работу модели

In [ ]:
def random_training_set():    
    chunk = random_chunk()
    inp = char_tensor(chunk[:-1])
    target = char_tensor(chunk[1:])
    return inp, target

Функция, которая выводит по строке-сиду предсказание работы модели

In [ ]:
def evaluate(prime_str='а', predict_len=100, temperature=0.8):
    hidden = decoder.init_hidden() # decoder - наша модель, ее переменная будет позже
    prime_input = char_tensor(prime_str) # кодируем входную строку
    predicted = prime_str # начало выходной строки
    predict_len -= len(prime_str)
    # Используем входную строку для задания hidden state
    for p in range(len(prime_str) - 1):
        _, hidden = decoder(prime_input[p], hidden)
    inp = prime_input[-1]
    
    for p in range(predict_len):
        output, hidden = decoder(inp, hidden)
        
        # Sample from the network as a multinomial distribution
        output_dist = output.data.view(-1).div(temperature).exp()
        top_i = torch.multinomial(output_dist, 1)[0]
        
        # Add predicted character to string and use as next input
        predicted_char = tokens[top_i]
        predicted += predicted_char
        inp = char_tensor(predicted_char)

    return predicted

Измерение времени

In [ ]:
import time, math

def time_since(since):
    s = time.time() - since
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

Основная функция обучения

In [ ]:
def train(inp, target):
    hidden = decoder.init_hidden()
    decoder.zero_grad()
    loss = 0

    for c in range(chunk_len):
        output, hidden = decoder(inp[c], hidden)
        # print(output, target)
        loss += criterion(output.squeeze(0), target[c]) # в исходной статье ошибочка, тут надо вытянуть в строку

    loss.backward()
    decoder_optimizer.step()

    return loss.item() / chunk_len

Тут создаем модель, так как по сути имеем задачу предсказания букв, что является многоклассовой классификацией, то используем функцию потери - кросс энтропия (`nn.CrossEntropyLoss()`), также оптимизатор `Adam` - стандартный для обучения нейросетей

In [ ]:
n_epochs = 2000
print_every = 100
plot_every = 10
hidden_size = 100
n_layers = 2
lr = 0.005
n_characters = len(tokens)
all_losses = []
decoder = RNN(n_characters, hidden_size, n_characters, n_layers) # собственно наша модель
decoder_optimizer = torch.optim.Adam(decoder.parameters(), lr=lr) # оптимищатор
criterion = nn.CrossEntropyLoss()

обучаем эпохи модели

In [ ]:
start = time.time()
loss_avg = 0
n_epochs = 3000
for epoch in range(1, n_epochs + 1):
    loss = train(*random_training_set()) # обучаем модель на сгенерированном батче       
    loss_avg += loss

    if epoch % print_every == 0:
        print('[%s (%d %d%%) %.4f]' % (time_since(start), epoch, epoch / n_epochs * 100, loss))
        print(evaluate(' мой дядя самых честных правил', 100), '\n')

    if epoch % plot_every == 0:
        all_losses.append(loss_avg / plot_every)
        loss_avg = 0

График обучения модели

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# %matplotlib inline

plt.figure()
plt.plot(all_losses)

In [ ]:
print(evaluate(' мой дядя самых честных правил', 500), '\n')

Пример текста сгенерированного обученной моделью доступен ниже. Не страшно, что в тексте много несуществующих слов. Используемая модель очень проста: это простая классическая RNN.

In [ ]:
for t in range(5, 7):
    print(evaluate(seed_phrase, 500, t / 10))
    print("________________________________")

Далее некоторые примеры работы сети, последний параметр - температура, чем меньше, тем более общепринятые слова/сочетания букв, чем больше, тем больше модель самостоятельно что-то предсказывает, и тем страннее может быть вывод. Поэтому берем примерно середину, что бы текст не состоял только из пробелов и предлогов, но и не был похож на странную последовательность рандомных букв и спецсимволов

In [ ]:
seed_phrase = " мой дядя самых честных правил"

In [ ]:
print(evaluate(seed_phrase, 500, 0.45))

In [ ]:
print(evaluate(seed_phrase, 500, 0.5))

In [ ]:
print(evaluate(seed_phrase, 500, 0.525))

In [ ]:
print(evaluate(seed_phrase, 500, 0.55))